In [23]:
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from plotly.subplots import make_subplots
import plotly.graph_objs as go
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, make_scorer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import RandomizedSearchCV
import shap
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
import plotly.express as px
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [2]:
RANDOM_STATE = 12345

In [3]:
df = pd.read_csv("../data/adult.csv")

In [4]:
X = df.drop('income', axis = 1)

In [5]:
y = df['income']

In [6]:
X.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States


In [7]:
y.value_counts()

income
<=50K    37155
>50K     11687
Name: count, dtype: int64

In [8]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(label_encoder.classes_)

['<=50K' '>50K']


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size = 0.2, random_state = RANDOM_STATE, stratify=y_encoded)

In [10]:
num_cols = X_train.select_dtypes(include='number').columns.tolist()

In [11]:
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

/var/folders/jt/pfrf2kqn59d1ph0418pyq4mr0000gn/T/ipykernel_54607/4180746908.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include='object').columns.tolist()


In [12]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

Функция оценки

In [13]:
def evaluate_pipeline(pipeline, X, y, cv):
    scoring = {"roc_auc": "roc_auc"}
    
    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )
    
    return scores["test_roc_auc"].mean()

In [14]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state = RANDOM_STATE)

Эксперимент 1 удаление столбца fnlwgt

In [15]:
X_no_weight = X.drop(columns=["fnlwgt"])

cat_cols = X_no_weight.select_dtypes(include="object").columns.tolist()
num_cols = X_no_weight.select_dtypes(exclude="object").columns.tolist()

preprocessor_no_weight = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

model = Pipeline([
    ("preprocessor", preprocessor_no_weight),
    ("model", RandomForestClassifier(random_state=42))
])

score_fnlwgt = evaluate_pipeline(model, X_no_weight, y_encoded, cv)
score_fnlwgt

/var/folders/jt/pfrf2kqn59d1ph0418pyq4mr0000gn/T/ipykernel_54607/1342789886.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_no_weight.select_dtypes(include="object").columns.tolist()


np.float64(0.8908066427583563)

Эксперимент 2 убрать education

In [16]:
X_no_edu = X.drop(columns=["education"])

cat_cols = X_no_edu.select_dtypes(include="object").columns.tolist()
num_cols = X_no_edu.select_dtypes(exclude="object").columns.tolist()

preprocessor_no_edu = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

model = Pipeline([
    ("preprocessor", preprocessor_no_edu),
    ("model", RandomForestClassifier(random_state=42))
])

score_edu = evaluate_pipeline(model, X_no_edu, y_encoded, cv)
score_edu

/var/folders/jt/pfrf2kqn59d1ph0418pyq4mr0000gn/T/ipykernel_54607/3153425779.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_no_edu.select_dtypes(include="object").columns.tolist()


np.float64(0.9048941129008249)

Эксперимент 3 добавить масштабирование числовых признаков

In [17]:
numeric_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor_scaled = ColumnTransformer([
    ("num", numeric_scaled, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

model = Pipeline([
    ("preprocessor", preprocessor_scaled),
    ("model", LogisticRegression(max_iter=2000))
])

score_scaled = evaluate_pipeline(model, X, y_encoded, cv)
score_scaled

np.float64(0.9066153868430046)

Эксперимент 4 использовать баланстировку классов

In [18]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

score_weight = evaluate_pipeline(model, X, y_encoded, cv)
score_weight

/Users/aleksejtolkunov/Desktop/my_project/Classification_python/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/aleksejtolkunov/Desktop/my_project/Classification_python/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also

np.float64(0.8923213893536239)

Эксперимент 5 SMOTE

In [24]:
model = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(max_iter=2000))
])

score_smote = evaluate_pipeline(model, X, y_encoded, cv)
score_smote

/Users/aleksejtolkunov/Desktop/my_project/Classification_python/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/aleksejtolkunov/Desktop/my_project/Classification_python/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also

np.float64(0.8932750994499029)

Сравнение экспериментов

In [27]:
results = pd.DataFrame({
    "experiment": [
        "drop_fnlwgt",
        "drop_education",
        "scaling",
        "class_weight",
        "SMOTE"
    ],
    "roc_auc": [
        score_fnlwgt,
        score_edu,
        score_scaled,
        score_weight,
        score_smote
    ]
})

results

,experiment,roc_auc
0,drop_fnlwgt,0.890807
1,drop_education,0.904894
2,scaling,0.906615
3,class_weight,0.892321
4,SMOTE,0.893275


Никакой из параметров значительно не повлиял на значение метрики, но лучше всего себя показал масштабирование числовых признаков